In [1]:
import os
import warnings

# Oculta el aviso de barras de progreso (no afecta los resultados)
warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, Reportes

In [2]:
DATASET = "ADAMlam-16/coffee-data_eda-project"
SPLIT = "train"

df = HFDatasetLoader(DATASET, split=SPLIT).load()
print("DataFrame original:", df.shape)
df.head()

DataFrame original: (10000, 16)


,ID,Age,Gender,Country,Coffee_Intake,Caffeine_mg,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Health_Issues,Occupation,Smoking,Alcohol_Consumption
0,1,40,Male,Germany,3.5,328.1,7.5,Good,24.9,78,Low,14.5,NaN,Other,0,0
1,2,33,Male,Germany,1.0,94.1,6.2,Good,20.0,67,Low,11.0,NaN,Service,0,0
2,3,42,Male,Brazil,5.3,503.7,5.9,Fair,22.7,59,Medium,11.2,Mild,Office,0,0
3,4,53,Male,Germany,2.6,249.2,7.3,Good,24.7,71,Low,6.6,Mild,Other,0,0
4,5,32,Female,Spain,3.1,298.0,5.3,Fair,24.1,76,Medium,8.5,Mild,Student,0,1


In [3]:
import pandas as pd

# 1. Selección de columnas: quitar ID (identificador) y Caffeine_mg (repite Coffee_Intake)
df_t = df.drop(columns=["ID", "Caffeine_mg"])
print("1. Columnas:", df.shape[1], "->", df_t.shape[1])

# 2. Limpieza de nulos: en Health_Issues, el nulo significa "sin problemas de salud"
nulos_antes = df_t["Health_Issues"].isna().sum()
df_t["Health_Issues"] = df_t["Health_Issues"].fillna("None")
print("2. Nulos en Health_Issues:", nulos_antes, "->", df_t["Health_Issues"].isna().sum())

# 3. Renombrar columnas al español
df_t = df_t.rename(columns={
    "Age": "Edad", "Gender": "Genero", "Country": "Pais",
    "Coffee_Intake": "Tazas_Cafe", "Sleep_Hours": "Horas_Sueno",
    "Stress_Level": "Nivel_Estres", "Occupation": "Ocupacion",
})
print("3. Columnas renombradas:", list(df_t.columns[:5]), "...")

# 4. Corrección de tipos: Smoking y Alcohol_Consumption son sí/no, no números
df_t[["Smoking", "Alcohol_Consumption"]] = df_t[["Smoking", "Alcohol_Consumption"]].astype(bool)
print("4. Tipos nuevos:", df_t[["Smoking", "Alcohol_Consumption"]].dtypes.to_dict())

# 5. Nueva columna: clasificar el consumo de café
df_t["Consumo_Cafe"] = pd.cut(
    df_t["Tazas_Cafe"], bins=[-0.1, 1, 3, float("inf")], labels=["Bajo", "Moderado", "Alto"]
)
print("5. Nueva columna Consumo_Cafe:\n", df_t["Consumo_Cafe"].value_counts())

print("\nDataFrame transformado:", df_t.shape)
df_t.head()

1. Columnas: 16 -> 14
2. Nulos en Health_Issues: 5941 -> 0
3. Columnas renombradas: ['Edad', 'Genero', 'Pais', 'Tazas_Cafe', 'Horas_Sueno'] ...
4. Tipos nuevos: {'Smoking': dtype('bool'), 'Alcohol_Consumption': dtype('bool')}
5. Nueva columna Consumo_Cafe:
 Consumo_Cafe
Moderado    4800
Alto        3498
Bajo        1702
Name: count, dtype: int64

DataFrame transformado: (10000, 15)


,Edad,Genero,Pais,Tazas_Cafe,Horas_Sueno,Sleep_Quality,BMI,Heart_Rate,Nivel_Estres,Physical_Activity_Hours,Health_Issues,Ocupacion,Smoking,Alcohol_Consumption,Consumo_Cafe
0,40,Male,Germany,3.5,7.5,Good,24.9,78,Low,14.5,None,Other,False,False,Alto
1,33,Male,Germany,1.0,6.2,Good,20.0,67,Low,11.0,None,Service,False,False,Bajo
2,42,Male,Brazil,5.3,5.9,Fair,22.7,59,Medium,11.2,Mild,Office,False,False,Alto
3,53,Male,Germany,2.6,7.3,Good,24.7,71,Low,6.6,Mild,Other,False,False,Moderado
4,32,Female,Spain,3.1,5.3,Fair,24.1,76,Medium,8.5,Mild,Student,False,True,Alto


In [4]:
reportes = Reportes()
reportes.imprimir_reporte(df_t)
ruta = reportes.guardar_reporte(df_t, "reportes/EA1_resumen_transformado.txt")
print("\nResumen guardado en:", ruta)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 15
Duplicados: 0

Columnas y tipos:
- Edad: int64
- Genero: str
- Pais: str
- Tazas_Cafe: float64
- Horas_Sueno: float64
- Sleep_Quality: str
- BMI: float64
- Heart_Rate: int64
- Nivel_Estres: str
- Physical_Activity_Hours: float64
- Health_Issues: str
- Ocupacion: str
- Smoking: bool
- Alcohol_Consumption: bool
- Consumo_Cafe: category

Valores nulos por columna:
- Edad: 0
- Genero: 0
- Pais: 0
- Tazas_Cafe: 0
- Horas_Sueno: 0
- Sleep_Quality: 0
- BMI: 0
- Heart_Rate: 0
- Nivel_Estres: 0
- Physical_Activity_Hours: 0
- Health_Issues: 0
- Ocupacion: 0
- Smoking: 0
- Alcohol_Consumption: 0
- Consumo_Cafe: 0

Estadísticas numéricas:
            Edad  Tazas_Cafe  Horas_Sueno        BMI  Heart_Rate  Physical_Activity_Hours
count  10000.000   10000.000    10000.000  10000.000   10000.000                10000.000
mean      34.949       2.509        6.636     23.987      70.618                    7.487
std       11.161       1.450        1.222      

## Transformaciones aplicadas

1. **Selección de columnas:** se eliminaron `ID`, que solo identifica cada fila, y `Caffeine_mg`, que repite la información de `Coffee_Intake` en otra unidad (correlación de 1,0).
2. **Limpieza de nulos:** `Health_Issues` tenía 5.941 valores vacíos. Se interpretaron como "sin problemas de salud" y se rellenaron con `"None"`.
3. **Renombrado de columnas:** las columnas principales se tradujeron al español para facilitar la lectura.
4. **Corrección de tipos:** `Smoking` y `Alcohol_Consumption` pasaron de números (0/1) a booleanos, porque representan sí/no.
5. **Nueva columna:** `Consumo_Cafe` clasifica el consumo diario en Bajo (hasta 1 taza), Moderado (más de 1 y hasta 3) y Alto (más de 3).

El resumen `.txt` se generó con la clase `Reportes` a partir del DataFrame transformado y se guardó en `reportes/EA1_resumen_transformado.txt`.